In [1]:
import sparknlp
spark = sparknlp.start(apple_silicon=True)
spark

25/08/29 10:20:41 WARN Utils: Your hostname, Js-MacBook-Air.local resolves to a loopback address: 127.0.0.1; using 192.168.36.15 instead (on interface en0)
25/08/29 10:20:41 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /Users/jefferyjapheth/.ivy2/cache
The jars for the packages stored in: /Users/jefferyjapheth/.ivy2/jars
com.johnsnowlabs.nlp#spark-nlp-silicon_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8f5bbdcd-58ac-4290-8b4f-b23b998eadfe;1.0
	confs: [default]
	found com.johnsnowlabs.nlp#spark-nlp-silicon_2.12;6.1.2 in central
	found com.typesafe#config;1.4.2 in central
	found org.rocksdb#rocksdbjni;6.29.5 in central
	found com.amazonaws#aws-java-sdk-s3;1.12.500 in central


:: loading settings :: url = jar:file:/Users/jefferyjapheth/miniconda3/envs/sparknlp/lib/python3.10/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found com.amazonaws#aws-java-sdk-kms;1.12.500 in central
	found com.amazonaws#aws-java-sdk-core;1.12.500 in central
	found commons-logging#commons-logging;1.1.3 in central
	found commons-codec#commons-codec;1.15 in central
	found org.apache.httpcomponents#httpclient;4.5.13 in central
	found org.apache.httpcomponents#httpcore;4.4.13 in central
	found software.amazon.ion#ion-java;1.0.2 in central
	found joda-time#joda-time;2.8.1 in central
	found com.amazonaws#jmespath-java;1.12.500 in central
	found com.github.universal-automata#liblevenshtein;3.0.0 in central
	found com.google.protobuf#protobuf-java-util;3.0.0-beta-3 in central
	found com.google.protobuf#protobuf-java;3.0.0-beta-3 in central
	found com.google.code.gson#gson;2.3 in central
	found it.unimi.dsi#fastutil;7.0.12 in central
	found org.projectlombok#lombok;1.16.8 in central
	found com.google.cloud#google-cloud-storage;2.20.1 in central
	found com.google.guava#guava;31.1-jre in central
	found com.google.guava#failureaccess;1.

In [2]:
# Load the cleaned contract dataset
df = spark.read.parquet("../data/processed/mcc_contracts")

print(f"Dataset loaded! Total records: {df.count():,}")
df.printSchema()

Dataset loaded! Total records: 343,849
root
 |-- contract: string (nullable = true)
 |-- description: string (nullable = true)
 |-- agreement_type: string (nullable = true)
 |-- type_score: string (nullable = true)
 |-- data_split: integer (nullable = true)
 |-- label_count: long (nullable = true)
 |-- type_label: string (nullable = true)



In [ ]:
# Step 1: Data Exploration and Basic Stats
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Fix type_score data type
df = df.withColumn("type_score", col("type_score").cast("float"))

# Show class distribution
print("Class Distribution:")
df.groupBy("agreement_type", "type_label").count().orderBy(desc("count")).show()

# Check data splits
print("Data Split Distribution:")
df.groupBy("data_split").count().show()

# Sample records to see the text
print("Sample Records:")
df.select("contract", "description", "agreement_type", "type_score").limit(3).show(truncate=False)

Class Distribution:


+---------------+----------+------+
| agreement_type|type_label| count|
+---------------+----------+------+
|     employment|   LABEL_1|144489|
|       security|   LABEL_0|102328|
|    purchase&ma|   LABEL_4| 47850|
|services&supply|   LABEL_3| 24882|
|    shareholder|   LABEL_5| 15716|
|          lease|   LABEL_2|  8584|
+---------------+----------+------+

Data Split Distribution:
+----------+-----+
|data_split|count|
+----------+-----+
|         1|68736|
|         5|68961|
|         2|68639|
|         3|68807|
|         4|68706|
+----------+-----+

Sample Records:
+-------------------------------+---------------------------------------------------------------+--------------+----------+
|contract                       |description                                                    |agreement_type|type_score|
+-------------------------------+---------------------------------------------------------------+--------------+----------+
|b53262vpexv10w24.txt           |form of amended and r

In [69]:
from sparknlp.base import DocumentAssembler, Finisher
from sparknlp.annotator import Tokenizer, Normalizer, LemmatizerModel, StopWordsCleaner
from pyspark.ml import Pipeline
from pyspark.sql.functions import explode, col, count, expr

In [79]:
from sparknlp.base import DocumentAssembler, Finisher
from sparknlp.annotator import Tokenizer, Normalizer, Stemmer, StopWordsCleaner
from pyspark.ml import Pipeline

# Document
document_assembler = DocumentAssembler() \
    .setInputCol("description") \
    .setOutputCol("document")

# Tokenize
tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("tokens")

# Normalize
normalizer = Normalizer() \
    .setInputCols(["tokens"]) \
    .setOutputCol("normalized_tokens") \
    .setLowercase(True)

# Stemmer (better for TF-IDF classification)
stemmer = Stemmer() \
    .setInputCols(["normalized_tokens"]) \
    .setOutputCol("stem_tokens")

# Stopwords
stopwords_cleaner = StopWordsCleaner() \
    .setInputCols(["stem_tokens"]) \
    .setOutputCol("clean_tokens") \
    .setCaseSensitive(False)

# Finisher
finisher = Finisher() \
    .setInputCols(["clean_tokens"]) \
    .setOutputCols(["finished_tokens"]) \
    .setOutputAsArray(True)

pipeline = Pipeline(stages=[
    document_assembler,
    tokenizer,
    normalizer,
    stemmer,
    stopwords_cleaner,
    finisher
])



In [80]:
nlp_model = pipeline.fit(df)
df_tokens = nlp_model.transform(df)

df_tokens = df_tokens.withColumn(
    "finished_tokens",
    expr("filter(finished_tokens, x -> length(x) > 2)")
)

# Check results
df_tokens.select("contract", "finished_tokens").show(20, truncate=False)


+-------------------------------+-------------------------------------------------------------+
|contract                       |finished_tokens                                              |
+-------------------------------+-------------------------------------------------------------+
|b53262vpexv10w24.txt           |[form, amend, restat, acquisitioncapit, line, credit]        |
|apc2015-12x178xkxexhibit101.htm|[amend, matur, extens, agreem, decemb]                       |
|creditagreement112701.htm      |[exhibit, credit, agreem]                                    |
|f10k123120_ex10z44.htm         |[exhibit, secur, replac, note, date, april]                  |
|dex105.htm                     |[secur, purchas, agreem]                                     |
|dex106.txt                     |[waiver, concern, secur, agreem]                             |
|c83266exv10we.txt              |[amend, credit, agreem]                                      |
|specimen_regswbba.htm          |[specim

**Cell 5: Apply Strategic Under-sampling**

In [81]:
# Cache tokenized data for performance
df_tokens = df_tokens.cache()
df_tokens.count()  # Trigger caching

# Define target counts for strategic under-sampling
target_counts = {
    'LABEL_1': 50000,  # employment (down from 144k)
    'LABEL_0': 35000,  # security (down from 102k)
    'LABEL_4': 25000,  # purchase&ma (down from 47k)
    'LABEL_3': 20000,  # services&supply (down from 24k)
    'LABEL_5': 15716,  # shareholder (keep all)
    'LABEL_2': 8584    # lease (keep all)
}

print("Applying strategic under-sampling...")
sampled_dfs = []
for label, target_count in target_counts.items():
    label_df = df_tokens.filter(col("type_label") == label)
    current_count = label_df.count()
    
    if current_count > target_count:
        fraction = target_count / current_count
        sampled_df = label_df.sample(withReplacement=False, fraction=fraction, seed=42)
        print(f"{label}: Sampled {target_count:,} from {current_count:,} ({fraction:.3f} fraction)")
    else:
        sampled_df = label_df
        print(f"{label}: Kept all {current_count:,}")
    
    sampled_dfs.append(sampled_df)

# Combine all sampled classes
balanced_df = sampled_dfs[0]
for df_part in sampled_dfs[1:]:
    balanced_df = balanced_df.union(df_part)

print(f"\nTotal records after under-sampling: {balanced_df.count():,}")
print("\nNew class distribution:")
balanced_df.groupBy("agreement_type", "type_label").count().orderBy(desc("count")).show()

Applying strategic under-sampling...
LABEL_1: Sampled 50,000 from 144,489 (0.346 fraction)
LABEL_0: Sampled 35,000 from 102,328 (0.342 fraction)
LABEL_4: Sampled 25,000 from 47,850 (0.522 fraction)
LABEL_3: Sampled 20,000 from 24,882 (0.804 fraction)
LABEL_5: Kept all 15,716
LABEL_2: Kept all 8,584

Total records after under-sampling: 154,255

New class distribution:
+---------------+----------+-----+
| agreement_type|type_label|count|
+---------------+----------+-----+
|     employment|   LABEL_1|49879|
|       security|   LABEL_0|34975|
|    purchase&ma|   LABEL_4|25012|
|services&supply|   LABEL_3|20089|
|    shareholder|   LABEL_5|15716|
|          lease|   LABEL_2| 8584|
+---------------+----------+-----+



**Cell 6: TF-IDF Feature Extraction**

In [82]:
from pyspark.ml.feature import HashingTF, IDF, StringIndexer

# Create TF-IDF pipeline
hashing_tf = HashingTF(
    inputCol="finished_tokens", 
    outputCol="raw_features", 
    numFeatures=10000  # Adjust based on vocabulary size needed
)

idf = IDF(
    inputCol="raw_features", 
    outputCol="tfidf_features",
    minDocFreq=5  # Filter rare terms
)

# Index labels for MLlib compatibility
label_indexer = StringIndexer(inputCol="type_label", outputCol="label_indexed")

# Build feature pipeline
feature_pipeline = Pipeline(stages=[hashing_tf, idf, label_indexer])

# Fit and transform
print("Extracting TF-IDF features...")
feature_model = feature_pipeline.fit(balanced_df)
feature_df = feature_model.transform(balanced_df)

print("Feature extraction complete!")
feature_df.select("contract", "agreement_type", "tfidf_features", "label_indexed").show(20)

Extracting TF-IDF features...
Feature extraction complete!
+--------------------+--------------+--------------------+-------------+
|            contract|agreement_type|      tfidf_features|label_indexed|
+--------------------+--------------+--------------------+-------------+
|          dex104.htm|    employment|(10000,[793,2389,...|          0.0|
|          dex102.htm|    employment|(10000,[1208,4264...|          0.0|
|   c95226exv10w1.htm|    employment|(10000,[793,6123,...|          0.0|
|   d379056dex101.htm|    employment|(10000,[4372,4549...|          0.0|
|         dex1022.htm|    employment|(10000,[564,5850,...|          0.0|
|  d274705dex1021.htm|    employment|(10000,[793,2084,...|          0.0|
|   d372980dex102.htm|    employment|(10000,[793,2410,...|          0.0|
|villageedocs06192...|    employment|(10000,[2752,3496...|          0.0|
|         dex1005.txt|    employment|(10000,[917,8203,...|          0.0|
|f8k120618ex10-1_c...|    employment|(10000,[63,3286,3...|       

**Cell 8: Calculate Class Weights**

In [55]:
import builtins  # to force Python's sum

# Get class counts
class_counts = feature_df.groupBy("label_indexed").count().collect()

# Use Python's built-in sum
total_samples = builtins.sum(row['count'] for row in class_counts)
n_classes = len(class_counts)

class_weights = {}
for row in class_counts:
    label = row['label_indexed']
    count = row['count']
    weight = total_samples / (n_classes * count)
    class_weights[label] = weight

print("Class weights for models:")
for label, weight in class_weights.items():
    print(f"Label {label}: {weight:.3f}")


Class weights for models:
Label 0.0: 0.515
Label 1.0: 0.735
Label 2.0: 1.028
Label 3.0: 1.280
Label 4.0: 1.636
Label 5.0: 2.995


**Cell 7: Prepare Train/Val/Test Splits**

In [83]:
from pyspark.sql.functions import col, desc

# Split based on pre-defined split column
train_df = feature_df.filter(col("data_split").isin([1, 2, 3]))
val_df   = feature_df.filter(col("data_split") == 4)
test_df  = feature_df.filter(col("data_split") == 5)

print("Data splits after balancing and feature extraction:")
print(f"Train: {train_df.count():,}")
print(f"Validation: {val_df.count():,}")
print(f"Test: {test_df.count():,}")

# Inspect class balance
print("\nTrain split class distribution:")
train_df.groupBy("agreement_type").count().orderBy(desc("count")).show(truncate=False)

print("\nValidation split class distribution:")
val_df.groupBy("agreement_type").count().orderBy(desc("count")).show(truncate=False)

print("\nTest split class distribution:")
test_df.groupBy("agreement_type").count().orderBy(desc("count")).show(truncate=False)


Data splits after balancing and feature extraction:
Train: 92,564
Validation: 30,624
Test: 31,067

Train split class distribution:
+---------------+-----+
|agreement_type |count|
+---------------+-----+
|employment     |29946|
|security       |20876|
|purchase&ma    |15088|
|services&supply|12011|
|shareholder    |9505 |
|lease          |5138 |
+---------------+-----+


Validation split class distribution:
+---------------+-----+
|agreement_type |count|
+---------------+-----+
|employment     |9913 |
|security       |6954 |
|purchase&ma    |4950 |
|services&supply|4041 |
|shareholder    |3064 |
|lease          |1702 |
+---------------+-----+


Test split class distribution:
+---------------+-----+
|agreement_type |count|
+---------------+-----+
|employment     |10020|
|security       |7145 |
|purchase&ma    |4974 |
|services&supply|4037 |
|shareholder    |3147 |
|lease          |1744 |
+---------------+-----+



In [84]:
from pyspark.sql.functions import create_map, lit, col
from itertools import chain
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

# ---- Assign class weights (native Spark, no UDF)
class_weight_map = create_map(
    [lit(float(x)) for x in chain.from_iterable(class_weights.items())]
)
train_df = train_df.withColumn("class_weight", class_weight_map[col("label_indexed")])

# ---- Logistic Regression model
lr = LogisticRegression(
    featuresCol="tfidf_features",
    labelCol="label_indexed",
    weightCol="class_weight",
    maxIter=100,
    regParam=0.05,
    elasticNetParam=0.1
)

# ---- Train
lr_model = lr.fit(train_df)

# ---- Validate
val_predictions = lr_model.transform(val_df)
evaluator = MulticlassClassificationEvaluator(
    labelCol="label_indexed",
    predictionCol="prediction",
    metricName="f1"
)
print("Validation F1:", evaluator.evaluate(val_predictions))


25/08/29 13:06:09 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS


Validation F1: 0.8613356178935974


In [85]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

for metric in ["f1", "precisionByLabel", "recallByLabel"]:
    evaluator = MulticlassClassificationEvaluator(
        labelCol="label_indexed",
        predictionCol="prediction",
        metricName=metric
    )
    print(metric, evaluator.evaluate(val_predictions))


f1 0.8613356178935974
precisionByLabel 0.9645969498910676
recallByLabel 0.8932714617169374


In [86]:
labels = feature_df.select("label_indexed").distinct().orderBy("label_indexed").collect()
evaluator = MulticlassClassificationEvaluator(
    labelCol="label_indexed",
    predictionCol="prediction"
)
for row in labels:
    label = row["label_indexed"]
    r = evaluator.setMetricName("recallByLabel").evaluate(val_predictions, {evaluator.metricLabel: label})
    p = evaluator.setMetricName("precisionByLabel").evaluate(val_predictions, {evaluator.metricLabel: label})
    print(f"Label {label}: precision={p:.3f}, recall={r:.3f}")


Label 0.0: precision=0.965, recall=0.893
Label 1.0: precision=0.952, recall=0.800
Label 2.0: precision=0.824, recall=0.820
Label 3.0: precision=0.590, recall=0.889
Label 4.0: precision=0.865, recall=0.846
Label 5.0: precision=0.959, recall=0.899


**SparkNLP LegalBERT Classifier Pipeline**

In [27]:
from sparknlp.base import *
from sparknlp.annotator import *
from pyspark.ml import Pipeline

# Document Assembler
document = DocumentAssembler() \
    .setInputCol("contract") \
    .setOutputCol("document")

# Tokenizer
tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("token")

# Pretrained LegalBERT embeddings
embeddings = BertEmbeddings.pretrained("legal_bert_base_uncased", "en") \
    .setInputCols(["document", "token"]) \
    .setOutputCol("embeddings") \
    .setCaseSensitive(False)

# Sentence embeddings (pool word embeddings → sentence/document vector)
sentence_embeddings = SentenceEmbeddings() \
    .setInputCols(["document", "embeddings"]) \
    .setOutputCol("sentence_embeddings") \
    .setPoolingStrategy("AVERAGE")

# Neural classifier
classifier = ClassifierDLApproach() \
    .setInputCols(["sentence_embeddings"]) \
    .setOutputCol("class") \
    .setLabelColumn("type_label") \
    .setBatchSize(64) \
    .setMaxEpochs(20) \
    .setLr(1e-3) \
    .setEnableOutputLogs(True)

# Build SparkNLP pipeline
nlp_pipeline = Pipeline(stages=[
    document,
    tokenizer,
    embeddings,
    sentence_embeddings,
    classifier
])


legal_bert_base_uncased download started this may take some time.


25/08/29 11:10:37 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


Approximate size to download 388.4 MB
[ | ]

25/08/29 11:10:38 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.
25/08/29 11:10:38 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


legal_bert_base_uncased download started this may take some time.
Approximate size to download 388.4 MB
[ — ]Download done! Loading the resource.
[ | ]Using CPUs
[OK!]


In [ ]:
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

# Explode tokens into separate rows
tokens_df = df_tokens.select(
    "agreement_type", 
    explode("finished_tokens").alias("token")
)

# Count token frequency per agreement_type
token_counts_df = tokens_df.groupBy("agreement_type", "token") \
    .count()

# Rank tokens by frequency per agreement_type
window_spec = Window.partitionBy("agreement_type").orderBy(col("count").desc())

ranked_tokens_df = token_counts_df.withColumn("rank", row_number().over(window_spec)) \
    .filter(col("rank") <= 50) \
    .orderBy("agreement_type", "rank")

ranked_tokens_df.show(100, truncate=False)


In [ ]:
# Pivot to see top tokens per agreement_type side by side
pivot_df = ranked_tokens_df.groupBy("rank") \
    .pivot("agreement_type") \
    .agg(first("token")) \
    .orderBy("rank")

pivot_df.show(truncate=False)


In [ ]:
from pyspark.sql import functions as F

# Step 1: Count in how many agreement types each token appears
token_type_counts = ranked_tokens_df.groupBy("token") \
    .agg(F.countDistinct("agreement_type").alias("type_count"))

# Step 2: Join back with original ranked_tokens_df
token_uniqueness_df = ranked_tokens_df.join(token_type_counts, on="token", how="left") \
    .withColumn("uniqueness", 
                F.when(F.col("type_count") == 1, F.lit("unique"))
                 .otherwise(F.lit("common"))
               ) \
    .select("agreement_type", "token", "count", "rank", "uniqueness") \
    .orderBy("agreement_type", "rank")

# Step 3: Show the result
token_uniqueness_df.show(100, truncate=False)


In [ ]:
from pyspark.sql.functions import explode, col, regexp_extract_all

# Example: collect all top tokens into a single pattern per agreement_type
token_list_df = tokens_df.groupBy("agreement_type") \
    .agg(F.collect_list("token").alias("tokens"))

# Convert list to a single regex pattern
token_list_df = token_list_df.withColumn(
    "pattern",
    F.concat_ws("|", "tokens")  # token1|token2|...
)

# Join to main DF
df_pattern = df.join(token_list_df, on="agreement_type")

# Use Spark SQL regex function
df_matches = df_pattern.withColumn(
    "matched_tokens",
    F.expr("regexp_extract_all(lower(description), pattern)")
)

# Explode and count frequencies
phrase_counts_df = df_matches.select(
    "agreement_type", explode("matched_tokens").alias("token")
).groupBy("agreement_type", "token") \
 .count() \
 .orderBy("agreement_type", col("count").desc())


In [ ]:
from pyspark.sql import functions as F
from sparknlp.base import DocumentAssembler
from sparknlp.annotator import Tokenizer, Normalizer, StopWordsCleaner
from pyspark.ml import Pipeline

# 1. Prepare the text column
document_assembler = DocumentAssembler() \
    .setInputCol("description") \
    .setOutputCol("document")

tokenizer = Tokenizer() \
    .setInputCols(["document"]) \
    .setOutputCol("token")

normalizer = Normalizer() \
    .setInputCols(["token"]) \
    .setOutputCol("normalized") \
    .setLowercase(True)

stop_cleaner = StopWordsCleaner() \
    .setInputCols(["normalized"]) \
    .setOutputCol("cleanTokens") \
    .setCaseSensitive(False)

# Pipeline
pipeline = Pipeline(stages=[
    document_assembler,
    tokenizer,
    normalizer,
    stop_cleaner
])

processed_df = pipeline.fit(df).transform(df)

# 2. Explode tokens
tokens_df = processed_df.select(
    "agreement_type",
    F.explode("cleanTokens.result").alias("token")
)

# 3. Count token frequency per type
freq_df = tokens_df.groupBy("agreement_type", "token").count()

# 4. Compute global frequency
global_df = freq_df.groupBy("token").agg(F.sum("count").alias("global_count"))

# 5. Join to compute uniqueness score
uniqueness_df = freq_df.join(global_df, on="token") \
    .withColumn("uniqueness", F.col("count") / F.col("global_count")) \
    .orderBy(F.desc("uniqueness"))

# 6. Get top N unique tokens per agreement_type
from pyspark.sql.window import Window

window = Window.partitionBy("agreement_type").orderBy(F.desc("uniqueness"))
top_keywords_df = uniqueness_df.withColumn("rank", F.row_number().over(window)) \
    .filter(F.col("rank") <= 5) \
    .select("agreement_type", "token", "count", "global_count", "uniqueness", "rank")

top_keywords_df.show(100, truncate=False)


In [ ]:
spark.stop()